In [123]:
"""
Parse ANADDB output file
Currently able to parse: EO tensor, Raman tensor, polarities.
"""
from itertools import tee
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt

### Reading the file ###
class Anaddb:
    def __init__(self, filename, natom=10):
        with open(filename, 'r') as file:
            self.data = file.readlines()
        self.natom = natom
        
    def ionic_EO(self):
        ionic_EO_tensor = np.zeros((6,3))
        index = self.data.index(" Total EO tensor (pm/V) in Voigt notations\n")+1
        
        for i in range(6):
            ionic_EO_tensor[i] = np.array(self.data[index+i].split(), dtype=float)
            
        return ionic_EO_tensor
    
    def get_EO_tensor(self):
        EO_tensors = pd.DataFrame(columns=["Mode number", "Frequency", "EO tensor"])
        index = self.data.index(" Output of the EO tensor (pm/V) in Voigt notations\n")+6
        
        for mode_number in range(4, (self.natom*3) + 1):
            ### space + digit + . + digit?
            frequency = float(re.findall("\s\d+.\d+", self.data[index])[0])
            
            EO = np.zeros((6, 3))
            for i in range(6):
                EO[i] = np.array(self.data[index+i+1].split(), dtype = float)
                
            EO_tensors.loc[len(EO_tensors.index)] = [mode_number, frequency, EO]
            index +=8
        return EO_tensors
            

    def get_Raman(self, NAC=False):
        Raman_tensors = pd.DataFrame(columns=["Mode", "Freq", "Raman susceptibility"])

        if NAC:
            index = self.data.index(" Raman susceptibility of zone-center phonons, with non-analyticity in the\n")+23
        else:
            index = self.data.index("  Raman susceptibilities of transverse zone-center phonon modes\n")+22

        for mode in np.arange(4, self.natom*3+1):
            freq = float(re.findall("\s\d+.\d+", self.data[index])[0])

            Raman = np.zeros((3,3))
            for i in range(3):
                Raman[i] = np.array(self.data[index+i+1][1:].split(), dtype=float)

            Raman_tensors.loc[len(Raman_tensors.index)] = [mode, freq, Raman]
            index += 6

        return Raman_tensors


    def get_polarity(self):
        polarity = pd.DataFrame(columns=["Mode", "px", "py", "pz", "|p|"])
        index = self.data.index(" Mode effective charges \n")+5
        
        for mode in np.arange(4, self.natom*3+1):
            polarity.loc[len(polarity.index)] = self.data[index][1:].split()
            index += 1

        return polarity
    
    
    def get_disp_mag(self):
        displacement = pd.DataFrame(columns=["Mode", "displacement set"])
        index = self.data.index(" Treat the first list of vectors \n")+93
        
        for mode in range(4, self.natom*3 + 1):
            disp = np.zeros((10, 3))
            
            for i in range(10):
                disp[i] = np.array(self.data[index+2*i][6:].split(), dtype=float) 
            
            displacement.loc[len(displacement.index)] = [mode, disp]
            index += 21
            
        return displacement

    # def get_disp_mag(self):
    #     displacement = pd.DataFrame(columns=["Mode", "Li1", "Li2", "Nb1", "Nb2", "O1", "O2", "O3", "O4", "O5", "O6"])
    #     index = self.data.index("  Treat the first list of vectors \n")+93
        
    #     for mode in range(4, self.natom*3 + 1):
    #         Li_1 = np.array(self.data[index+1].split(), dtype=float)
    #         Li_2 = np.array(self.data[index+3].split(), dtype=float)
    #         Nb_1 = np.array(self.data[index+5].split(), dtype=float)
    #         Nb_2 = np.array(self.data[index+7].split(), dtype=float)
            
            
        
            
    #         displacement.loc[len(displacement.index)] = [mode, Li_1, Li_2, Nb_1, Nb_2, O_1, O_2, O_3, O_4, O_5, O_6]
    #         index += 23
            
    #     return displacement
    
    
    def plot_r_33(self, filename):
        EO_tensors = self.get_EO()
        r_33 = [i[2,2] for i in EO_tensors["EO Tensor"]]
        # inv_freq_sq = 1 / np.array(EO_tensors["Freq"])**2

        fig, ax1 = plt.subplots()
        ax2 = ax1.twinx()
        fig.set_figwidth(15)
        fig.set_figheight(5)

        ax1.plot(EO_tensors["Mode"], r_33, 'b-')
        ax2.plot(EO_tensors["Mode"], EO_tensors["Freq"], 'r-')
        ax1.set_xlabel("phonon mode number")
        ax1.set_ylabel("r_33 (pm/V)", color="b")
        ax2.set_ylabel("mode frequency (cm-1)", color="r")
        ax1.set_title("Mode Decomposed EO Tensor")
        plt.savefig(filename)


    def plot_Raman(self, filename, NAC=False):
        Raman_tensors = self.get_Raman()
        EO_tensors = self.get_EO()
        r_33 = [i[2,2] for i in EO_tensors["EO Tensor"]]
        Raman_33 = [np.abs(i[2,2]) for i in Raman_tensors["Raman susceptibility"]]

        fig, ax1 = plt.subplots()
        ax2 = ax1.twinx()
        fig.set_figwidth(15)
        fig.set_figheight(5)

        ax1.plot(Raman_tensors["Mode"], r_33, 'b-')
        ax2.plot(Raman_tensors["Mode"], Raman_33, 'r-')
        # ax1.vlines(x=41, ymin=0, ymax=8)
        ax1.set_xlabel("phonon mode number")
        ax1.set_ylabel("r_33 (pm/V)", color="b")
        ax2.set_ylabel("|Raman_33|", color="r")
        ax1.set_title("Mode Decomposed EO Tensor")
        plt.savefig(filename)


    def plot_polarity(self, filename):
        polarity = self.get_polarity()
        EO_tensors = self.get_EO()
        r_33 = [i[2,2] for i in EO_tensors["EO Tensor"]]
        p = [float(i) for i in polarity["|p|"]]
        modes = [float(i) for i in polarity["Mode"]]

        fig, ax1 = plt.subplots()
        ax2 = ax1.twinx()
        fig.set_figwidth(15)
        fig.set_figheight(5)

        ax1.plot(modes, r_33, 'b-')
        ax2.plot(modes, p, 'r-')
        # ax1.vlines(x=41, ymin=0, ymax=8)
        ax1.set_xlabel("phonon mode number")
        ax1.set_ylabel("r_33 (pm/V)", color="b")
        ax2.set_ylabel("|polarity|", color="r")
        ax1.set_title("Mode Decomposed EO Tensor")
        plt.savefig(filename)
    

In [124]:
if __name__ == "__main__":
    LNO = Anaddb("tnlo_4.abo")
    # BTO.plot_Raman("Raman_33.pdf")
    print(LNO.ionic_EO())

[[  0.           4.78801885 -10.07651212]
 [  0.          -4.78801885 -10.07651212]
 [ -0.           0.         -27.6903241 ]
 [ -0.         -15.36850141   0.        ]
 [-15.36850141  -0.          -0.        ]
 [  4.78801885   0.          -0.        ]]


In [132]:
LNO.get_disp_mag().loc[0, 'displacement set']

array([[ 1.83718752e-04, -1.15007327e-03,  0.00000000e+00],
       [ 1.84057570e-04,  1.15001910e-03,  0.00000000e+00],
       [ 5.07575147e-04,  7.25753224e-04,  0.00000000e+00],
       [ 5.07361309e-04, -7.25902731e-04,  0.00000000e+00],
       [ 7.45343876e-04,  1.18279121e-03, -1.49249231e-03],
       [ 1.17002938e-03,  1.00614683e-03,  5.63959501e-05],
       [ 1.11066515e-03,  1.46225746e-03,  1.43609636e-03],
       [ 7.45712504e-04, -1.18241641e-03,  1.49225756e-03],
       [ 1.11095143e-03, -1.46199027e-03, -1.43635975e-03],
       [ 1.17045003e-03, -1.00589715e-03, -5.58978101e-05]])